# Example Notebook: Analysing the *Can-WeTrust-LLMs-GeneratedCode* Dataset

This notebook provides a simple, reusable starting point for analysing the dataset of LLM-generated code.

## What this notebook does
- loads the dataset from CSV
- inspects schema and missing values
- summarizes prompts, models, languages, vulnerabilities, and code smells
- plots frequency distributions
- computes simple security-risk summaries
- exports a small summary report

## Expected dataset fields
This notebook is designed to work with columns such as:

- `id`
- `prompt`
- `prompt_type`
- `language`
- `llm_model`
- `generated_code`
- `vulnerabilities`
- `code_smells`
- `security_score`
- `analysis_notes`

It is written defensively, so it will still run if some columns are missing.

In [ ]:
# If running in Google Colab, mount Drive if your dataset is stored there.
# You can skip this cell if your CSV file is uploaded directly to Colab.

RUNNING_IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    RUNNING_IN_COLAB = True
except Exception:
    RUNNING_IN_COLAB = False

if RUNNING_IN_COLAB:
    print("Google Colab detected.")
    # Uncomment the next line if you want to mount Google Drive.
    # drive.mount('/content/drive')
else:
    print("Running outside Google Colab.")

In [ ]:
# Install dependencies if needed
# In many Colab environments, pandas/matplotlib are already available.

# !pip install pandas matplotlib numpy

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Set the dataset path

Update the path below to match your dataset file.
Examples:

- `/content/dataset_description.csv`
- `/content/drive/MyDrive/Can-WeTrust-LLMs-GeneratedCode/metadata/dataset_description.csv`

In [ ]:
DATASET_PATH = "/content/dataset_description.csv"  # <-- change this

if not os.path.exists(DATASET_PATH):
    print(f"Dataset file not found: {DATASET_PATH}")
    print("Please update DATASET_PATH to your CSV file location.")
else:
    print(f"Found dataset: {DATASET_PATH}")

In [ ]:
# Load dataset
if os.path.exists(DATASET_PATH):
    df = pd.read_csv(DATASET_PATH)
    print("Dataset loaded successfully.")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
else:
    df = pd.DataFrame()
    print("Empty DataFrame created because dataset path was not found.")

## 2. Initial inspection

In [ ]:
if not df.empty:
    display(df.head())
    print("\nColumns:")
    print(list(df.columns))

In [ ]:
if not df.empty:
    print("Data types:\n")
    print(df.dtypes)

In [ ]:
if not df.empty:
    missing = df.isna().sum().sort_values(ascending=False)
    missing_df = pd.DataFrame({
        "missing_count": missing,
        "missing_percent": (missing / len(df) * 100).round(2)
    })
    display(missing_df)

## 3. Helper functions

In [ ]:
def normalize_multivalue_cell(value):
    '''
    Convert a vulnerability/code-smell cell into a list.
    Supports separators like commas, semicolons, pipes, and new lines.
    '''
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    parts = re.split(r"[,\n;|]+", text)
    return [p.strip() for p in parts if p.strip()]

def safe_value_counts(dataframe, column_name):
    if column_name not in dataframe.columns:
        return pd.Series(dtype="int64")
    return dataframe[column_name].astype(str).fillna("NA").value_counts()

def plot_bar(series, title, xlabel, ylabel="Count", top_n=15, rotation=45):
    if series.empty:
        print(f"No data available for: {title}")
        return
    s = series.head(top_n)
    plt.figure(figsize=(10, 5))
    plt.bar(s.index.astype(str), s.values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    plt.show()

## 4. Basic dataset statistics

In [ ]:
if not df.empty:
    print(f"Total samples: {len(df)}")

    if "language" in df.columns:
        print(f"Unique languages: {df['language'].nunique(dropna=True)}")
    if "llm_model" in df.columns:
        print(f"Unique models: {df['llm_model'].nunique(dropna=True)}")
    if "prompt_type" in df.columns:
        print(f"Unique prompt types: {df['prompt_type'].nunique(dropna=True)}")

In [ ]:
if not df.empty and "language" in df.columns:
    lang_counts = df["language"].fillna("Unknown").value_counts()
    display(lang_counts.to_frame("count"))
    plot_bar(lang_counts, "Samples per Programming Language", "Language")

In [ ]:
if not df.empty and "llm_model" in df.columns:
    model_counts = df["llm_model"].fillna("Unknown").value_counts()
    display(model_counts.to_frame("count"))
    plot_bar(model_counts, "Samples per LLM Model", "Model")

In [ ]:
if not df.empty and "prompt_type" in df.columns:
    prompt_counts = df["prompt_type"].fillna("Unknown").value_counts()
    display(prompt_counts.to_frame("count"))
    plot_bar(prompt_counts, "Samples per Prompt Type", "Prompt Type")

## 5. Vulnerability analysis

In [ ]:
if not df.empty and "vulnerabilities" in df.columns:
    vuln_lists = df["vulnerabilities"].apply(normalize_multivalue_cell)
    all_vulns = [item for sublist in vuln_lists for item in sublist]
    vuln_counts = pd.Series(all_vulns).value_counts() if all_vulns else pd.Series(dtype="int64")

    print(f"Total labelled vulnerability mentions: {len(all_vulns)}")
    print(f"Unique vulnerability labels: {len(vuln_counts)}")

    display(vuln_counts.head(20).to_frame("count"))
    plot_bar(vuln_counts, "Top Vulnerabilities", "Vulnerability", top_n=15, rotation=60)
else:
    print("Column 'vulnerabilities' not found.")

In [ ]:
if not df.empty and {"language", "vulnerabilities"}.issubset(df.columns):
    temp = df.copy()
    temp["has_vulnerability"] = temp["vulnerabilities"].fillna("").astype(str).str.strip().ne("")
    language_vuln_rate = temp.groupby("language")["has_vulnerability"].mean().sort_values(ascending=False) * 100
    display(language_vuln_rate.round(2).to_frame("vulnerability_rate_percent"))

    plt.figure(figsize=(10, 5))
    plt.bar(language_vuln_rate.index.astype(str), language_vuln_rate.values)
    plt.title("Percentage of Samples with Vulnerabilities by Language")
    plt.xlabel("Language")
    plt.ylabel("Percent")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Code smell analysis

In [ ]:
if not df.empty and "code_smells" in df.columns:
    smell_lists = df["code_smells"].apply(normalize_multivalue_cell)
    all_smells = [item for sublist in smell_lists for item in sublist]
    smell_counts = pd.Series(all_smells).value_counts() if all_smells else pd.Series(dtype="int64")

    print(f"Total labelled code smell mentions: {len(all_smells)}")
    print(f"Unique code smell labels: {len(smell_counts)}")

    display(smell_counts.head(20).to_frame("count"))
    plot_bar(smell_counts, "Top Code Smells", "Code Smell", top_n=15, rotation=60)
else:
    print("Column 'code_smells' not found.")

## 7. Security score analysis

In [ ]:
if not df.empty and "security_score" in df.columns:
    df["security_score"] = pd.to_numeric(df["security_score"], errors="coerce")
    print(df["security_score"].describe())

    plt.figure(figsize=(8, 5))
    plt.hist(df["security_score"].dropna(), bins=20)
    plt.title("Distribution of Security Scores")
    plt.xlabel("Security Score")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    print("Column 'security_score' not found.")

In [ ]:
if not df.empty and {"prompt_type", "security_score"}.issubset(df.columns):
    score_by_prompt = df.groupby("prompt_type")["security_score"].mean().sort_values(ascending=False)
    display(score_by_prompt.round(2).to_frame("average_security_score"))

    plt.figure(figsize=(8, 5))
    plt.bar(score_by_prompt.index.astype(str), score_by_prompt.values)
    plt.title("Average Security Score by Prompt Type")
    plt.xlabel("Prompt Type")
    plt.ylabel("Average Security Score")
    plt.tight_layout()
    plt.show()

In [ ]:
if not df.empty and {"llm_model", "security_score"}.issubset(df.columns):
    score_by_model = df.groupby("llm_model")["security_score"].mean().sort_values(ascending=False)
    display(score_by_model.round(2).to_frame("average_security_score"))

    plt.figure(figsize=(10, 5))
    plt.bar(score_by_model.index.astype(str), score_by_model.values)
    plt.title("Average Security Score by LLM Model")
    plt.xlabel("LLM Model")
    plt.ylabel("Average Security Score")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 8. Cross-analysis: language × prompt type

In [ ]:
if not df.empty and {"language", "prompt_type"}.issubset(df.columns):
    cross_tab = pd.crosstab(df["language"], df["prompt_type"])
    display(cross_tab)

    cross_tab.plot(kind="bar", figsize=(10, 6))
    plt.title("Language vs Prompt Type")
    plt.xlabel("Language")
    plt.ylabel("Number of Samples")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Optional: length-based analysis of generated code

In [ ]:
if not df.empty and "generated_code" in df.columns:
    df["code_length_chars"] = df["generated_code"].fillna("").astype(str).apply(len)
    print(df["code_length_chars"].describe())

    plt.figure(figsize=(8, 5))
    plt.hist(df["code_length_chars"], bins=20)
    plt.title("Distribution of Generated Code Length")
    plt.xlabel("Characters")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    print("Column 'generated_code' not found.")

## 10. Export summary tables

In [ ]:
OUTPUT_DIR = Path("./analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not df.empty:
    summary = {
        "total_samples": len(df),
        "num_columns": len(df.columns),
        "languages": int(df["language"].nunique()) if "language" in df.columns else None,
        "models": int(df["llm_model"].nunique()) if "llm_model" in df.columns else None,
        "prompt_types": int(df["prompt_type"].nunique()) if "prompt_type" in df.columns else None,
        "mean_security_score": float(df["security_score"].mean()) if "security_score" in df.columns else None,
    }

    pd.DataFrame([summary]).to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)

    if "language" in df.columns:
        df["language"].value_counts().to_csv(OUTPUT_DIR / "language_counts.csv", header=["count"])

    if "llm_model" in df.columns:
        df["llm_model"].value_counts().to_csv(OUTPUT_DIR / "model_counts.csv", header=["count"])

    if "prompt_type" in df.columns:
        df["prompt_type"].value_counts().to_csv(OUTPUT_DIR / "prompt_type_counts.csv", header=["count"])

    print(f"Summary files exported to: {OUTPUT_DIR.resolve()}")
else:
    print("No dataset loaded. Nothing exported.")

## 11. Suggested research questions you can analyse with this notebook

1. Which programming language shows the highest proportion of vulnerable generated samples?
2. Does few-shot prompting reduce the average security risk score?
3. Which LLM produces fewer vulnerability labels on average?
4. What code smells recur most often across models and languages?
5. Is longer generated code associated with higher security risk?

You can extend this notebook with:
- CWE family grouping
- severity distribution analysis
- per-model/per-language heatmaps
- hypothesis testing
- publication-ready figures